## [ Video RAG] Retrieve; & Generation

In [1]:
%pip install -Uqqq langchain langchain-openai langchain-pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.2 MB/s eta 0:00:00


In [2]:
# 구글 드라이브 마운트 및 뎐동
from google.colab import drive # Colab에서 구글드라이브 사용 모듈
drive.mount('/content/drive') # 현재 런타임에서 내 구글드라이브 마운트 (연동)

BASE_PATH = '/content/drive/MyDrive/skn_34/05_multimodal_rag'

Mounted at /content/drive


In [3]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')

## Chain 구성

In [26]:
from typing import Optional # Optional 타입 (선택값)
from pydantic import BaseModel, Field # 구조화된 응답 스키마 정의
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_pinecone import PineconeVectorStore

# 비디오 검색 결과에 대한 답변 구조(스키마) 클래스
class VideoAnswer(BaseModel):
  found: bool = Field(description='적절한 비디오 검색 결과를 찾았는지 여부')
  answer: str = Field(description='질문에 대해 검색 결과 기반의 답변 내용')
  video_filename: Optional[str] = Field(description='참조한 비디오 파일명')
  frame_no: Optional[str] = Field(description='참조한 프레임 번호')
  frame_caption: Optional[str] = Field(description='참조한 프레임 설명')

embedding = OpenAIEmbeddings(model='text-embedding-3-small')

vector_store = PineconeVectorStore(
    index_name='video-caption-kr',
    embedding=embedding
)

llm = init_chat_model('gpt-5.6-luna')

prompt = PromptTemplate.from_template('''
당신은 비디오 장면 검색 에이전트입니다.
제공된 비디오 검색결과에서 사용자의 질문에 적합한 장면을 찾고, 답변해주세요.
제공된 비디오 검색결과에서 사용자의 질문에 적합한 장면이 없다면, 검색결과가 없다고 답변해주세요.

출력 규칙(매우 중요):
- frame_no는 반드시 "단 하나의 프레임 번호"만 출력한다.
- frame_no에는 숫자만 포함한다. (예: "37")
- 여러 후보가 있으면 가장 적합한 1개만 선택한다.
- 하나로 결정할 수 없으면 found=false로 하고, answer에 "단일 프레임으로 결정 불가"라고 적는다.
- video_filename도 반드시 단 하나만 출력한다.

[사용자 질문]
{question}

[참조할 비디오 검색결과(프레임 설명)]
{context}
''')

chain = prompt | llm.with_structured_output(VideoAnswer)

## 헬퍼함수
- RAG 검색결과 컨텍스트 구성
- 프레임/비디오 HTML 함수

In [31]:
import base64
from IPython.display import HTML # 노트북
import os

# 검색 결과 문서 리스트(docs)를 LLM 입력용 context 문자열로 합쳐주는 함수
def build_context(docs):
  context = ""

  for i, doc in enumerate(docs, 1):
    metadata = doc.metadata
    context += f"""
    [{i}]
    비디오: {metadata['video_filename']}
    프레임: {metadata['frame_no']}
    해설: {doc.page_content}
    """
  return context # 전체 문서를 하나의 문자열로 반환

# 주어진 비디오 파일명/프레임 번호로 프레임 이미지 파일의 전체 경로를 반환하는 함수
def get_frame_image_path(video_filename, frame_no, frame_dir='frames'):
  video_basename = os.path.splitext(video_filename)[0]
  frame_no = int(float(frame_no)) # 실수로 들어오는 경우 대비 str -> float -> int
  filename = f'{video_basename}_{frame_no:05d}.jpg' # 5자리 0패딩 포함한 파일명
  return os.path.join(BASE_PATH, frame_dir, video_basename, filename) # 전체 파일 경로

# 비디오 파일을 base64로 인코딩해서 HTML <video> 태그로 출력하는 함수
def display_video(video_filename, video_dir='videos', width=300):
  video_path = os.path.join(BASE_PATH, video_dir, video_filename) # 동영상 파일 경로

  with open(video_path, 'rb') as f:
    b64_str = base64.b64encode(f.read()).decode('utf-8') # Base64 인코딩 -> 문자열 디코딩

  data_url = f'data:video/mp4;base64,{b64_str}' # HTML URL 형식으로 구성

  html = HTML(f'''
<video width='{width}' controls>
  <source src='{data_url}' type='video/mp4'/>
</video>
''')
  display(html)

# 이미지 파일을 출력하는 함수
def display_image(frame_image_path, width=300):
  with open(frame_image_path, 'rb') as f:
    b64_str = base64.b64encode(f.read()).decode('utf-8')

  html = f'''
<img src='data:image/jpeg;base64,{b64_str}' width='{width}'/>
'''
  display(HTML(html))

## MutiModal RAG
- RAG Chain
- RAG Agent

In [32]:
# 사용자 질의(query)로 유사 프레임을 찾은 후, LLM이 구조화된 답변을 만들고, 이 결과를 시각화해서 출력하는 함수
def video_search_and_answer(query, k=3):
  # 1. 벡터스토어 검색
  docs = vector_store.similarity_search(query, k=k)

  # 2. 체인 실행 (프롬프트를 이용한 증강)
  context = build_context(docs) # 검색된 문서를 하나의 문자열로 합침
  response = chain.invoke({'question': query, 'context': context}) # 구조화된 답변 출력 chain

  # 3. 응답 VideoAnswer 객체를 포멧팅
  if response.found:
    video_filename = response.video_filename
    frame_no = response.frame_no
    print('비디오 검색 결과를 찾았습니다.')
    print('[비디오 플레이어]')
    display_video(video_filename) # HTML video 태그 플레이어 렌더링
    print('[관련 프레임]')
    frame_image_path = get_frame_image_path(video_filename, frame_no)
    print(f"[비디오 파일명] {response.video_filename}")
    print(f"[프레임 번호] {response.frame_no}")
    print(f"[답변] {response.answer}")
  else:
    print('검색된 비디오/프레임 정보가 없습니다.')

In [33]:
video_search_and_answer('축구공을 트래핑하는 장면이 있어?')

Output hidden; open in https://colab.research.google.com to view.

In [34]:
video_search_and_answer('설원에서 스키타는 장면있어?')

Output hidden; open in https://colab.research.google.com to view.